# Nightly Price Modeling V3

I keep the price model notebook clear and choose one winner only. In this notebook I benchmark single models only, avoid ensemble confusion, and choose the model with the highest pricing score by R? while still showing the MAE trade-off.


## Setup Note

In this cell I import the shared Gold ML helpers and the sklearn utilities needed for the pricing benchmark and final model save step.


In [2]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

cwd = Path.cwd().resolve()

candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    cwd / "code files" / "ml",
    cwd.parent / "ml",
    cwd.parent / "code files" / "ml",
]

for candidate in candidates:
    if (candidate / "gold_ml_modeling.py").exists():
        sys.path.insert(0, str(candidate))
        print("Added:", candidate)
        break
else:
    raise FileNotFoundError("Could not find gold_ml_modeling.py")


from gold_ml_modeling import (
    MODEL_DIR,
    PRICE_CATEGORICAL_COLUMNS,
    PRICE_LAUNCH_CORE_FEATURES,
    PRICE_MARKET_PROXY_FEATURES,
    add_full_price_proxies,
    add_train_only_price_proxies,
    build_candidate_models,
    build_mysql_engine,
    build_pipeline,
    load_property_mart_frame,
    prepare_common_features,
)

MODEL_DIR


Added: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml


WindowsPath('C:/Users/mario/Documents/DSMLAISL LEARNING PATH/Portugal-Rental-Investment-/code files/data/gold/modeling')

## Data Load Note

In this cell I load the latest Gold property mart and prepare the shared engineered features. The target here is the current nightly price because I want a recommended asking price model for the product.


In [3]:
engine = build_mysql_engine()
model_df = load_property_mart_frame(engine)
prepared_df = prepare_common_features(model_df)
prepared_df = prepared_df.dropna(subset=["target_nightly_price"]).copy()
prepared_df.shape


(10005, 71)

## Feature Design Note

In this cell I define the pricing feature sets. I keep a launch-friendly core, then add market price anchors, then add quality and observed operating context so I can push the pricing score higher while staying explicit about each layer.


In [4]:
quality_columns = [
    "host_is_superhost",
    "platform_count",
    "number_of_reviews",
    "reviews_per_month",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "review_scores_value",
    "host_response_rate",
    "host_acceptance_rate",
    "host_listings_count",
    "host_total_listings_count",
    "host_tenure_days",
    "host_tenure_years",
    "review_score_blend",
]

availability_columns = [
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
]

financing_columns = [
    "financing_vintage_year",
    "applied_asset_discount_pct",
    "applied_annual_interest_rate",
]

price_feature_sets = {
    "launch_pricing_core": PRICE_LAUNCH_CORE_FEATURES,
    "launch_plus_market_proxies": PRICE_LAUNCH_CORE_FEATURES + PRICE_MARKET_PROXY_FEATURES,
    "launch_plus_market_quality": PRICE_LAUNCH_CORE_FEATURES + PRICE_MARKET_PROXY_FEATURES + quality_columns,
    "observed_market_full": PRICE_LAUNCH_CORE_FEATURES + PRICE_MARKET_PROXY_FEATURES + quality_columns + availability_columns + financing_columns,
}

price_feature_sets.keys()


dict_keys(['launch_pricing_core', 'launch_plus_market_proxies', 'launch_plus_market_quality', 'observed_market_full'])

## Benchmark Note

In this cell I benchmark the pricing candidates across both raw-price and log-price targets. I compare only single models, and I always evaluate back in euros so the scores stay interpretable.


In [5]:
candidate_models = build_candidate_models(include_linear=True, include_dummy=True)
candidate_models = {
    name: model
    for name, model in candidate_models.items()
    if name in {"dummy_median", "ridge", "random_forest", "hist_gradient_boosting", "catboost", "xgboost"}
}

X = prepared_df.drop(columns=["target_nightly_price"])
y = prepared_df["target_nightly_price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test = add_train_only_price_proxies(X_train, X_test, y_train)

price_rows = []
fitted_price_models = {}
for target_mode in ["raw_price", "log_price"]:
    y_train_fit = np.log1p(y_train) if target_mode == "log_price" else y_train.copy()

    for feature_set_name, feature_columns in price_feature_sets.items():
        for model_name, model in candidate_models.items():
            pipeline = build_pipeline(feature_columns, PRICE_CATEGORICAL_COLUMNS, clone(model))
            pipeline.fit(X_train[feature_columns], y_train_fit)
            predictions = pipeline.predict(X_test[feature_columns])
            if target_mode == "log_price":
                predictions = np.expm1(predictions)
            predictions = np.clip(predictions, 0, None)

            price_rows.append({
                "target_mode": target_mode,
                "feature_set": feature_set_name,
                "model_name": model_name,
                "mae": mean_absolute_error(y_test, predictions),
                "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
                "r2": r2_score(y_test, predictions),
            })
            fitted_price_models[(target_mode, feature_set_name, model_name)] = (pipeline, feature_columns)

price_benchmark_df = pd.DataFrame(price_rows).sort_values(["mae", "rmse", "r2"], ascending=[True, True, False]).reset_index(drop=True)
price_benchmark_df.to_csv(MODEL_DIR / "nightly_price_model_benchmark_v3.csv", index=False)
price_benchmark_df.head(20)


,target_mode,feature_set,model_name,mae,rmse,r2
0,log_price,observed_market_full,xgboost,26.524553,39.275703,0.712058
1,log_price,launch_plus_market_quality,xgboost,26.623829,39.595281,0.707353
2,log_price,observed_market_full,hist_gradient_boosting,27.167904,40.397538,0.695374
3,log_price,launch_plus_market_quality,hist_gradient_boosting,27.300663,40.570840,0.692754
4,raw_price,observed_market_full,xgboost,27.543863,39.461611,0.709325
5,raw_price,launch_plus_market_quality,xgboost,27.575835,39.839010,0.703739
6,log_price,observed_market_full,catboost,27.617671,40.925633,0.687357
7,log_price,launch_plus_market_quality,catboost,27.806536,41.114277,0.684468
8,raw_price,observed_market_full,hist_gradient_boosting,28.071912,40.443940,0.694674
9,raw_price,launch_plus_market_quality,hist_gradient_boosting,28.445653,40.941504,0.687115


## Final Choice Note

In this cell I make the pricing decision explicit. I choose the single model with the highest R? and keep the best MAE row visible as a trade-off reference so the choice is transparent.


In [ ]:
best_r2_row = price_benchmark_df.sort_values("r2", ascending=False).iloc[0]
best_mae_row = price_benchmark_df.sort_values("mae", ascending=True).iloc[0]
best_r2_row, best_mae_row


(target_mode                 log_price
 feature_set      observed_market_full
 model_name     hist_gradient_boosting
 mae                         27.127757
 rmse                        40.082326
 r2                           0.692691
 Name: 2, dtype: object,
 target_mode                     log_price
 feature_set    launch_plus_market_quality
 model_name                        xgboost
 mae                             26.944658
 rmse                            40.153601
 r2                               0.691598
 Name: 0, dtype: object)

## Final Training Note

In this cell I refit the chosen pricing winner on the full latest dataset and save a clean final bundle. If the winner uses the log-price target, I save that information in the artifact so the later prediction layer knows how to transform outputs back into euros.


In [ ]:
chosen_target_mode = str(best_r2_row["target_mode"])
chosen_feature_set = str(best_r2_row["feature_set"])
chosen_model_name = str(best_r2_row["model_name"])
chosen_feature_columns = price_feature_sets[chosen_feature_set]

full_df = add_full_price_proxies(prepared_df.copy(), "target_nightly_price")
final_pipeline = build_pipeline(chosen_feature_columns, PRICE_CATEGORICAL_COLUMNS, clone(candidate_models[chosen_model_name]))
y_fit = np.log1p(full_df["target_nightly_price"]) if chosen_target_mode == "log_price" else full_df["target_nightly_price"]
final_pipeline.fit(full_df[chosen_feature_columns], y_fit)

price_model_path = MODEL_DIR / "nightly_price_model_final_v3.joblib"
price_metadata_path = MODEL_DIR / "nightly_price_model_final_v3_metadata.json"
joblib.dump({
    "pipeline": final_pipeline,
    "feature_columns": chosen_feature_columns,
    "target_column": "target_nightly_price",
    "target_mode": chosen_target_mode,
    "chosen_model_name": chosen_model_name,
    "chosen_feature_set": chosen_feature_set,
}, price_model_path)
price_metadata_path.write_text(json.dumps({
    "notebook": "nightly_price_modeling_v3.ipynb",
    "chosen_winner_by_r2": best_r2_row.to_dict(),
    "best_mae_row": best_mae_row.to_dict(),
}, indent=2), encoding="utf-8")

price_model_path, price_metadata_path


(WindowsPath('C:/Users/mario/Documents/DSMLAISL LEARNING PATH/Portugal-Rental-Investment-/code files/data/gold/modeling/nightly_price_model_final_v3.joblib'),
 WindowsPath('C:/Users/mario/Documents/DSMLAISL LEARNING PATH/Portugal-Rental-Investment-/code files/data/gold/modeling/nightly_price_model_final_v3_metadata.json'))